# BAC × SegEarth-R1: full reproduction on Kaggle
Animesh Kumar · Preyanshi Doshi · School of Electrical Engineering, MIT Manipal

**Before running:** Settings → *Accelerator* = **GPU T4 x2** · *Internet* = **On** (needs a phone-verified account) · Add Input → your `bac-code` dataset.
Set `MODE` below (`quick` ≈ 1 h: core claims; `full` ≈ 4–6 h: all 29 configurations), then **Save Version → Save & Run All (Commit)** so it runs in the background. Results appear in the *Output* tab.

In [ ]:
MODE = 'full'           # 'quick' (~1 h, core claims) or 'full' (all 29 configurations)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys, torch; print(sys.version.split()[0], '| torch', torch.__version__, '| CUDA GPUs:', torch.cuda.device_count())" 

## 1. Install the pinned libraries (PyTorch comes preinstalled on Kaggle)

In [ ]:
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'    # PyTorch only (Kaggle's JAX needs NumPy 2; we keep Kaggle's NumPy)
!pip -q install transformers==4.36.2 tokenizers==0.15.2 timm==0.6.13 fvcore addict einops einops-exts==0.0.4 accelerate==0.21.0 peft==0.4.0 tabulate 2>&1 | tail -2
!python -c "import numpy, transformers, timm, torch; print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__, '| torch', torch.__version__)" 

## 2. Unpack the project code (from the `bac-code` dataset)

In [ ]:
import glob, os, shutil, zipfile
os.makedirs('/kaggle/working/Minor', exist_ok=True)
zips = glob.glob('/kaggle/input/**/bac_code.zip', recursive=True)
if zips:                                    # dataset kept the zip
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/Minor')
else:                                       # Kaggle auto-extracted the zip: copy the extracted tree
    marker = glob.glob('/kaggle/input/**/bac/common.py', recursive=True)
    assert marker, 'Add the bac-code dataset via Add Input'
    root = os.path.dirname(os.path.dirname(marker[0]))
    shutil.copytree(root, '/kaggle/working/Minor', dirs_exist_ok=True)
os.chdir('/kaggle/working/Minor')
!ls

## 3. Download the released checkpoint (3.2 GB) and EarthReason train + val (~0.9 GB)

In [ ]:
# downloads run in a subprocess so the pinned libraries are used everywhere
!python -c "from huggingface_hub import snapshot_download; snapshot_download('earth-insights/SegEarth-R1-EarthReason', local_dir='checkpoints/SegEarth-R1-EarthReason')"
!python -c "from huggingface_hub import hf_hub_download; import zipfile; [zipfile.ZipFile(hf_hub_download('earth-insights/EarthReason', f'{s}.zip', repo_type='dataset', local_dir='/kaggle/tmp')).extractall('data/rs_reason_seg/RSReasonSeg') for s in ('val', 'train')]"
!ls data/rs_reason_seg/RSReasonSeg/train/images | wc -l; ls data/rs_reason_seg/RSReasonSeg/val/images | wc -l; du -sh checkpoints

In [ ]:
# fail fast: if the environment is broken, stop here instead of skipping through hours of runs
import subprocess, json, os, sys
r = subprocess.run('python bac/evaluate.py --name smoke --limit 5 --out /kaggle/tmp/smoke', shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
assert os.path.exists('/kaggle/tmp/smoke/smoke_val.json'), 'SMOKE TEST FAILED: evaluation does not run - see error above'
print('smoke eval OK, gIoU on 5 images:', round(json.load(open('/kaggle/tmp/smoke/smoke_val.json'))['gIoU'], 3))

In [ ]:
r = subprocess.run('python bac/train.py --name smoke --kind bac --tokens 64 --steps 3', shell=True, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-3000:])
assert os.path.exists('runs/smoke/connector.pt'), 'SMOKE TEST FAILED: training does not run - see error above'
import shutil; shutil.rmtree('runs/smoke'); print('smoke train OK')

## 4. Contract tests (all five connectors, every budget, gradient flow)

In [ ]:
!python bac/test_connectors.py 2>&1 | grep -v -i "warn\|pytree\|redirect" 

## 5. Smoke test: 10 images, checks the model loads and runs on the GPU and shows the speed

In [ ]:
!python bac/evaluate.py --name smoke --limit 10 --out /kaggle/tmp/smoke 2>&1 | grep -E "^\[smoke\]|gIoU\"|Error" 

## 6. Run the experiments (skips anything already done, so it is safe to re-run)

In [ ]:
!bash kaggle/kaggle_queue.sh {MODE}

## 7. Analyse and verify against the reference results from the Apple M4

In [ ]:
!python bac/analyze.py > results/analyze.txt 2>&1; head -40 results/analyze.txt
!python bac/verify_against_reference.py --ref reference_results --new results

## 8. Package everything for download (Output tab → `bac_kaggle_results.zip`)

In [ ]:
!cd /kaggle/working/Minor && zip -qr /kaggle/working/bac_kaggle_results.zip results figures runs -x 'runs/*/connector.pt' && ls -la /kaggle/working/bac_kaggle_results.zip
!cat results/verification_report.md | head -30
# keep the saved output small: drop the downloaded weights and dataset (re-downloadable)
!rm -rf /kaggle/working/Minor/checkpoints /kaggle/working/Minor/data /kaggle/tmp